# 03 — Experiment Results

## Where results live

Each search run appends to `outputs/ledger/<run>.jsonl` — one JSON line per node.
A sidecar `outputs/ledger/<run>.manifest.json` records the exact data-selection and
search-contract knobs (split seed, fractions, allowed levers, eta, etc.) so the run is
reproducible.

This notebook loads a ledger and lets you:
- Browse all nodes as a DataFrame
- Inspect any individual node (params, spec_config, scores)
- Visualise the Ladder climb (best val score over search steps)
- Re-run the exact same search offline to verify reproducibility

**Set `RUN_PATH` in the Setup cell to point at a real run. If the file does not exist,
a deterministic demo ledger is generated automatically.**

In [ ]:
import sys, os, warnings
import matplotlib.pyplot as plt
import pandas as pd

# --- PARAMETERS (edit these) ---
RUN_PATH = "outputs/ledger/demo.jsonl"
# --------------------------------

from scrye.ledger import Ledger
from scrye.tree_view import tree_lines
from scrye.results import (
    nodes_frame, node_report, format_node_report,
    best_progression, spec_from_node, replay_proposals,
)
from scrye.search import GatePolicy, Proposal, SearchContract, run_search
from scrye.spec import PipelineSpec

if not os.path.exists(RUN_PATH):
    print("[DEMO] RUN_PATH not found — generating a demo ledger with a stub score_fn.")
    print("[DEMO] Point RUN_PATH at a real run file to use real data.\n")

    os.makedirs(os.path.dirname(RUN_PATH) if os.path.dirname(RUN_PATH) else ".", exist_ok=True)

    # Deterministic stub: increasing dev/val scores, two real registry levers
    _proposals_list = [
        Proposal("recalib.global_temp", {"T": 1.5}, "Flatten over-confident instruct distributions"),
        Proposal("recalib.dirichlet", {"alpha": 0.05}, "Add smoothing toward uniform to lift under-predicted tails"),
    ]
    _dev_scores = [42.3, 45.1]
    _val_scores = [43.8, 46.5]
    _call_counts = {"dev": 0, "val": 0}

    def _stub_score_fn(spec, recs):
        if recs == ["val"]:
            i = min(_call_counts["val"], len(_val_scores) - 1)
            _call_counts["val"] += 1
            return _val_scores[i], {"cost_usd": 0.001}
        else:
            i = min(_call_counts["dev"], len(_dev_scores) - 1)
            _call_counts["dev"] += 1
            return _dev_scores[i], {"cost_usd": 0.002}

    _it = iter(_proposals_list)
    _demo_ledger = Ledger(RUN_PATH)
    _contract = SearchContract(
        goal="Improve SimBench S score on dev+val",
        allowed_levers=["recalib.global_temp", "recalib.dirichlet"],
        autonomy_budget=10,
        val_query_budget=5,
        gate_policy=GatePolicy(gate_on_surprise=False),
        eta=0.5,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        run_search(
            _contract, _demo_ledger,
            dev=["dev"], val=["val"],
            score_fn=_stub_score_fn,
            propose_fn=lambda state: next(_it, None),
            now_fn=lambda: "2026-06-20T00:00:00",
        )
    print(f"[DEMO] Demo ledger written to {RUN_PATH} ({len(_demo_ledger.nodes)} nodes).\n")

    # Write a companion manifest so the reproducibility cell demonstrates
    # data-selection and parameter recording (not just the replay check).
    from scrye.manifest import RunManifest, dataset_fingerprint
    from scrye.levers import REGISTRY_VERSION

    _manifest_path = RUN_PATH.replace(".jsonl", ".manifest.json")
    # Note: split values below are illustrative — the demo uses a stub scorer, not a real dataset split.
    print("[DEMO] Split fields (split_seed, split_fractions, etc.) are illustrative; "
          "the demo uses a stub scorer, not a real dataset split.")
    _manifest = RunManifest(
        registry_version=REGISTRY_VERSION,
        split_seed=0,
        split_fractions={"dev": 0.5, "val": 0.25, "test": 0.25},
        split_unit="question",
        pin_required_to="test",
        goal=_contract.goal,
        allowed_levers=_contract.allowed_levers,
        autonomy_budget=_contract.autonomy_budget,
        val_query_budget=_contract.val_query_budget,
        eta=_contract.eta,
        cost_cap_usd=getattr(_contract, "cost_cap_usd", 100.0),
        model="gemini-flash-lite",
        temperature=0.0,
        seed=0,
        dataset_fingerprint=dataset_fingerprint([]),
        created_at="2026-06-20T00:00:00",
    )
    _manifest.save(_manifest_path)
    print(f"[DEMO] Demo manifest written to {_manifest_path}.\n")


In [ ]:
ledger = Ledger.load(RUN_PATH)
print(f"Loaded {len(ledger.nodes)} nodes, global_k={ledger.global_k()}, "
      f"best_val={ledger.best_val():.4f}, total_cost=${ledger.total_cost():.4f}\n")
print("Search tree:")
print("\n".join(tree_lines(ledger)))


In [ ]:
df = nodes_frame(ledger)
display(df)

# Scatter: dev_score vs val_score, coloured by accepted status
val_rows = df[df["val_score"].notna()].copy()
if not val_rows.empty:
    colors = val_rows["accepted"].map({True: "tab:green", False: "tab:red", None: "tab:gray"})
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(val_rows["dev_score"], val_rows["val_score"], c=colors, s=80, edgecolors="k", linewidths=0.5)
    ax.set_xlabel("Dev score")
    ax.set_ylabel("Val score")
    ax.set_title("Dev vs Val score (green=accepted, red=rejected)")
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="tab:green", label="accepted"),
        Patch(facecolor="tab:red", label="rejected"),
        Patch(facecolor="tab:gray", label="no val query"),
    ]
    ax.legend(handles=legend_elements)
    plt.tight_layout()
    plt.show()
else:
    print("No val-queried nodes to plot.")


In [ ]:
# Change NODE_ID to inspect a different node
NODE_ID = ledger.nodes[-1].node_id  # last node by default

print(format_node_report(ledger, NODE_ID))


In [ ]:
bp = best_progression(ledger)
display(bp)

if not bp.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(bp["step"], bp["best_so_far"], marker="o", color="steelblue", label="best_so_far")
    ax.scatter(bp["step"], bp["val_score"], color="lightgray", zorder=3, label="val_score (all)")
    # Mark new-best points
    new_best = bp[bp["is_new_best"]]
    ax.scatter(new_best["step"], new_best["best_so_far"], color="gold",
               s=120, zorder=4, edgecolors="darkorange", linewidths=1.5, label="new best")
    ax.set_xlabel("Step (val query index)")
    ax.set_ylabel("Val score")
    ax.set_title("Ladder climb — best val score over search steps")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No val-queried nodes for progression plot.")


## Reproducibility

### How to reproduce this run

The sidecar `<run>.manifest.json` records every knob that was deterministic at search time:
split seed/fractions/unit, allowed levers, eta, model, temperature, and a dataset fingerprint.

To reproduce:

1. **Re-select the data** — call `make_split(records, seed=manifest.split_seed, fractions=manifest.split_fractions, unit=manifest.split_unit, pin_required_to=manifest.pin_required_to)` to get identical dev/val/test sets.
2. **Replay the proposals** — `replay_proposals(ledger)` returns the exact sequence of `Proposal(lever_id, params, rationale)` objects recorded in the ledger.
3. **Re-run search** — pass a fresh `Ledger`, the same `SearchContract`, and a `propose_fn` that exhausts the replayed list.

**Why it's identical:** cached LLM calls (temp=0 + on-disk cache) are byte-for-byte repeatable. Uncached calls with temp=0 are near-identical but not guaranteed by the API. The split and proposal sequence are pure functions of the recorded seeds.

The cell below runs an **offline reproducibility check** using the same stub score function and asserts that the replayed ledger matches the original.

In [ ]:
# Load manifest if present
import os, json as _json
manifest_path = RUN_PATH.replace(".jsonl", ".manifest.json")
if os.path.exists(manifest_path):
    from scrye.manifest import RunManifest
    _manifest = RunManifest.load(manifest_path)
    print("Run manifest:")
    for k, v in _json.loads(open(manifest_path).read()).items():
        print(f"  {k}: {v}")
else:
    print("(No manifest file found.)")

print()

# --- Offline reproducibility check ---
# Re-run the SAME stub search with replay_proposals feeding the proposer
# into a fresh ledger, then assert lever sequence and global_k match.

_replayed_proposals = replay_proposals(ledger)
_replay_iter = iter(_replayed_proposals)

# Use the same deterministic stub score function
_rc = {"dev": 0, "val": 0}
_dscores = [42.3, 45.1]
_vscores = [43.8, 46.5]

def _replay_score_fn(spec, recs):
    if recs == ["val"]:
        i = min(_rc["val"], len(_vscores) - 1)
        _rc["val"] += 1
        return _vscores[i], {"cost_usd": 0.001}
    else:
        i = min(_rc["dev"], len(_dscores) - 1)
        _rc["dev"] += 1
        return _dscores[i], {"cost_usd": 0.002}

import tempfile as _tmp
_td = _tmp.mkdtemp()
_replay_ledger = Ledger(os.path.join(_td, "replay.jsonl"))
_replay_contract = SearchContract(
    goal="Improve SimBench S score on dev+val",
    allowed_levers=["recalib.global_temp", "recalib.dirichlet"],
    autonomy_budget=10,
    val_query_budget=5,
    gate_policy=GatePolicy(gate_on_surprise=False),
    eta=0.5,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    run_search(
        _replay_contract, _replay_ledger,
        dev=["dev"], val=["val"],
        score_fn=_replay_score_fn,
        propose_fn=lambda state: next(_replay_iter, None),
        now_fn=lambda: "2026-06-20T00:00:00",
    )

# Assert lever sequence matches
orig_levers = [n.lever_id for n in ledger.nodes]
replay_levers = [n.lever_id for n in _replay_ledger.nodes]
assert orig_levers == replay_levers, f"Lever mismatch: {orig_levers} != {replay_levers}"
assert ledger.global_k() == _replay_ledger.global_k(), (
    f"global_k mismatch: {ledger.global_k()} != {_replay_ledger.global_k()}"
)

print(f"Original  : levers={orig_levers}, global_k={ledger.global_k()}")
print(f"Replayed  : levers={replay_levers}, global_k={_replay_ledger.global_k()}")
print("REPRODUCED ✓")
